# Classifier-free guidance on DeepFashion images

The base diffusion model for Modality 1 and its guidance demonstration. **The noise schedule,
U-Net, training loop, guided sampler, FID cells and sample grid are a teammate's
`CIS6720_Proj1_DiffusionCode_Modality_1.ipynb`, kept as written** (the copy received is in this
folder). What this notebook adds: the loader imported from `src/guidance/data/fashion.py` (the
same `FashionDataset`), a checkpoint saved where the three comparison notebooks find it, the
**classifier accuracy and precision / recall** evaluation of the flow-matching twin
(`CIS6720_Proj1_FlowMatchingCode_Modality_1.ipynb`, Experiment 3 and the cell after it), and a
manifest with the commit, config, hardware, training time and every number.

Conditioning is the 50-way garment category through an embedding added to the time embedding;
10% of labels are dropped during training so the null branch exists. Classifier-free guidance
at sampling: `eps = eps_uncond + w (eps_cond - eps_uncond)`, with `w = 1` plain conditional
sampling and `w = 0` unconditional.

**Three settings differ from the notebook as received.** Batch size 64 rather than 16: the
flow-matching twin uses 64, and this base is shared by four notebooks. The sampler has an `eta`
term added (DDIM eq. 16): `eta = 1`, the default, is ancestral DDPM sampling, which every
DeepFashion and cell-state notebook in this repo uses; `eta = 0` is their deterministic sampler
unchanged. And the checkpoint is saved under `checkpoints/diffusion_fashion/best.pt` in the
format the comparison notebooks load. Everything else, including the 50-epoch early-stopping
recipe, is theirs.

Run top to bottom on a GPU runtime. **Run this notebook first**; the three comparison notebooks
load its checkpoint. Any setting below can be overridden from the environment. For a run that
shows everything working in well under an hour, put `%env CFG_QUICK=1` in a cell above the
settings (here and in the three comparison notebooks): a class-balanced 10k of the 209k training
images, 8 epochs, 256 evaluation images. Its numbers are not results.

In [ ]:
# Bootstrap. Run once.
import importlib.util, subprocess, sys, os, time

for pkg, mod in (("torchmetrics", "torchmetrics"), ("torch-fidelity", "torch_fidelity")):
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

if importlib.util.find_spec("guidance") is None:
    # Inside a checkout (Jupyter on a server, cwd somewhere under the repo): walk up to its src/.
    from pathlib import Path as _P
    _src = next((p / "src" for p in [_P.cwd(), *_P.cwd().parents] if (p / "src" / "guidance").is_dir()), None)
    if _src is None:                                   # Colab: clone the repo next to the notebook
        if not os.path.exists("discrete-generative-models-proj-1"):
            subprocess.run(["git", "clone", "-q",
                            "https://github.com/imperorrp/discrete-generative-models-proj-1.git"],
                           check=True)
        else:                                          # cloned earlier next to this notebook: update it
            subprocess.run(["git", "-C", "discrete-generative-models-proj-1", "pull", "-q"], check=False)
        _src = "discrete-generative-models-proj-1/src"
    sys.path.insert(0, str(_src))
    print("guidance package from:", _src)

def _git_sha():
    for d in ("discrete-generative-models-proj-1", "."):
        try:
            return subprocess.check_output(["git", "-C", d, "rev-parse", "--short", "HEAD"],
                                           text=True, stderr=subprocess.DEVNULL).strip()
        except Exception:
            pass
    return "unknown"

GIT_SHA = _git_sha()
import torch
print("code version:", GIT_SHA, "| PyTorch:", torch.__version__)

## Settings

Theirs (`Imports and shared settings`), with environment overrides for headless runs and the two differences stated above.

In [ ]:
# Imports and shared settings
from pathlib import Path
import math
import random
import time
from collections import defaultdict

import numpy as np
from PIL import Image, ImageOps
import torch
from torch.utils.data import DataLoader, Dataset, Subset

def _env(name, default):
    v = os.environ.get(f"CFG_{name}")
    return default if v is None else type(default)(v)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = _env("SEED", 42)
image_size = 64
batch_size = _env("BATCH_SIZE", 64)          # as received: 16; the flow-matching twin and the comparison notebooks use 64
num_classes = 50  # The coarse DeepFashion category table has 50 names.
null_class = num_classes  # Extra embedding used when the label is dropped for CFG.
BASE_CHANNELS = _env("BASE_CHANNELS", 64)
# QUICK = 1 is the showcase budget: a class-balanced seeded 10k of the 209k training images, 8
# epochs, 256 evaluation images, 3 classifier epochs. The same switch in the comparison notebooks
# picks the same 10k images. A quick run shows that everything works; its numbers are not results.
QUICK = _env("QUICK", 0)
EPOCHS = _env("EPOCHS", 8 if QUICK else 50)
PATIENCE = _env("PATIENCE", 10)
LIMIT = _env("LIMIT", 10_000 if QUICK else 0)   # >0: a class-balanced seeded subset of the training images
VAL_LIMIT = _env("VAL_LIMIT", 2048)          # class-balanced validation subset
SAMPLE_STEPS = _env("SAMPLE_STEPS", 50)
ETA = _env("ETA", 1.0)                       # 1 = ancestral DDPM sampling (every notebook here); 0 = their deterministic sampler
GUIDANCE_SWEEP = [float(v) for v in os.environ.get("CFG_GUIDANCE_SWEEP", "0,1,3").split(",")]
N_EVAL = _env("N_EVAL", 256 if QUICK else 2048)   # generated images per guidance strength for FID, accuracy, precision / recall
CLF_EPOCHS = _env("CLF_EPOCHS", 3 if QUICK else 10)   # fine-tuning epochs of the scorer
REUSE_CKPT = _env("REUSE_CKPT", 1)           # load best.pt from OUT_DIR if present instead of retraining
NUM_WORKERS = _env("NUM_WORKERS", 2)         # loader workers; 0 on Windows or when run as a script
CKPT_DIR = Path(os.environ.get("FASHION_CKPT_DIR", "checkpoints"))
OUT_DIR = CKPT_DIR / "diffusion_fashion"
OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
print(f"Using device: {device} | epochs {EPOCHS} | batch {batch_size} | train images {LIMIT or 'all'} | sweep {GUIDANCE_SWEEP} | n_eval {N_EVAL}"
      + (" | QUICK showcase budget" if QUICK else ""))

## Data

Their `FashionDataset` and caption reader, imported from the package (identical code): crop to the
garment box, pad to a white square, 64×64, random horizontal flips for training, pixels in
`[-1, 1]`, the official train / validation partition, a class-balanced validation subset of 2048.
Put the benchmark zip, the image zip and `captions.csv` next to this notebook (or under `data/`);
this cell unzips them on first use, the image zip taking a few minutes, once. An already unpacked
folder holding `Anno_coarse/`, `Eval/` and `img/` is found in the same places; `FASHION_ROOT` and
`FASHION_CAPTIONS` point somewhere else.

In [ ]:
from guidance.data.fashion import FashionDataset, load_captions, find_root

data_root = find_root()
print("DeepFashion root:", data_root)
captions_by_image = load_captions(data_root)
print(f"Loaded {len(captions_by_image):,} captions")

# LIMIT > 0 (QUICK or smoke runs): FashionDataset's own class-balanced seeded subset, the rule their
# script applies to --max-train-images, so every class is still represented in a small run.
train_data = FashionDataset(data_root, "train", captions_by_image, image_size, flip=True, limit=LIMIT, seed=seed)
val_data = FashionDataset(data_root, "val", captions_by_image, image_size, limit=VAL_LIMIT, seed=seed)
active_class_ids = sorted({label for _, label, _ in train_data.records})
class_names = train_data.classes
if len(class_names) != num_classes:          # their assert; the real table has 50 names, a smoke tree fewer
    num_classes = null_class = len(class_names)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
print(f"Train: {len(train_data):,}; validation: {len(val_data):,}; "
      f"active classes: {len(active_class_ids)} of {num_classes}")

# A dataset item now contains its image, category, and caption together.
image, class_id, caption = train_data[0]
print(f"Example {class_names[class_id]} caption: {caption}")

## Noise schedule

Theirs, verbatim: a 1000-step cosine schedule with betas clamped to [1e-4, 0.9999], epsilon prediction.

In [ ]:
# Define the same 1,000-step cosine noise schedule as fashion_cfg_diffusion.py.
K = 1000
step_fraction = torch.linspace(0, K, K + 1, device=device, dtype=torch.float64) / K
cosine_curve = torch.cos((step_fraction + 0.008) / 1.008 * math.pi / 2).square()
cosine_curve = cosine_curve / cosine_curve[0]
betas = (1 - cosine_curve[1:] / cosine_curve[:-1]).clamp(0.0001, 0.9999).float()
alpha_bars = torch.cumprod(1 - betas, dim=0)  # Indices 0..999; no clean-image index.


def Add_Noise(x0, k):
    '''Draw Gaussian noise and construct x_k for one random timestep per image.'''
    epsilon = torch.randn_like(x0)
    alpha = alpha_bars[k][:, None, None, None]
    xk = alpha.sqrt() * x0 + (1 - alpha).sqrt() * epsilon
    return xk, epsilon

## Model

Theirs, verbatim. The class label is added to the time embedding and enters every residual block; bottleneck self-attention at 16×16.

In [ ]:
# Define the same conditional U-Net as fashion_cfg_diffusion.py, explicitly.
from torch import nn
import torch.nn.functional as F

base_channels = BASE_CHANNELS


def timestep_embedding(k, width):
    '''Sinusoidal timestep features, before a learned projection.'''
    half = width // 2
    frequencies = torch.exp(-math.log(10000) * torch.arange(half, device=k.device)
                            / max(half - 1, 1))
    angles = k.float()[:, None] * frequencies[None]
    result = torch.cat([angles.sin(), angles.cos()], dim=-1)
    return F.pad(result, (0, width - result.shape[-1]))


class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, embedding_width):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.embedding = nn.Linear(embedding_width, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip = (nn.Conv2d(in_channels, out_channels, 1)
                     if in_channels != out_channels else nn.Identity())

    def forward(self, x, embedding):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.embedding(F.silu(embedding))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Attention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.out = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        q, k, v = self.qkv(self.norm(x)).chunk(3, dim=1)
        q = q.flatten(2).transpose(1, 2)
        k = k.flatten(2).transpose(1, 2)
        v = v.flatten(2).transpose(1, 2)
        h = F.scaled_dot_product_attention(q, k, v)
        h = h.transpose(1, 2).reshape(batch, channels, height, width)
        return x + self.out(h)


class DiffusionUNet(nn.Module):
    def __init__(self, classes=num_classes, base=base_channels):
        super().__init__()
        self.base = base
        width = base * 4
        self.time_mlp = nn.Sequential(nn.Linear(base, width), nn.SiLU(), nn.Linear(width, width))
        self.class_embedding = nn.Embedding(classes + 1, width)
        self.input = nn.Conv2d(3, base, 3, padding=1)
        self.down1 = ResBlock(base, base, width)
        self.stride1 = nn.Conv2d(base, base, 3, stride=2, padding=1)
        self.down2 = ResBlock(base, base * 2, width)
        self.stride2 = nn.Conv2d(base * 2, base * 2, 3, stride=2, padding=1)
        self.middle1 = ResBlock(base * 2, base * 2, width)
        self.attention = Attention(base * 2)  # Bottleneck self-attention at 16×16.
        self.middle2 = ResBlock(base * 2, base * 2, width)
        self.up2 = ResBlock(base * 4, base * 2, width)
        self.up1 = ResBlock(base * 3, base, width)
        self.output_norm = nn.GroupNorm(8, base)
        self.output = nn.Conv2d(base, 3, 3, padding=1)
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, xk, k, class_labels):
        embedding = (self.time_mlp(timestep_embedding(k, self.base))
                     + self.class_embedding(class_labels))
        x = self.input(xk)
        skip1 = self.down1(x, embedding)              # [B, base, 64, 64]
        skip2 = self.down2(self.stride1(skip1), embedding)  # [B, 2*base, 32, 32]
        x = self.middle2(self.attention(self.middle1(self.stride2(skip2), embedding)), embedding)
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.up2(torch.cat([x, skip2], dim=1), embedding)
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.up1(torch.cat([x, skip1], dim=1), embedding)
        return self.output(F.silu(self.output_norm(x)))  # Predict RGB noise.


model = DiffusionUNet().to(device)
N_PARAMS = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {N_PARAMS:,}")

## Training

Theirs, with two lines added: the epoch of the best validation loss and a per-epoch history are
recorded for the checkpoint and the manifest. AdamW 2e-4, constant learning rate, mixed
precision, gradient clip 1, 10% label dropout, up to 50 epochs with early stopping after 10
without a significant improvement; validation reuses the same noise and timesteps every epoch.

If `checkpoints/diffusion_fashion/best.pt` already exists (upload it to skip training), it is
loaded after a check that it was trained under this setup, and the loop below runs zero epochs.
Whichever path ran, everything below uses the best-validation weights.

In [ ]:
CKPT_META = dict(split=f"DeepFashion official train/val; validation subset {VAL_LIMIT} class-balanced, seed {seed}",
                 n_classes=num_classes, image_size=image_size, base=base_channels, num_train_timesteps=K,
                 drop_class_percentage=0.1, n_params=N_PARAMS)
LOADED, history, best_epoch, TRAIN_SECONDS = False, [], 0, 0.0
if REUSE_CKPT and (OUT_DIR / "best.pt").exists():
    ck = torch.load(OUT_DIR / "best.pt", map_location=device, weights_only=False)
    mismatch = {k: (ck.get("meta", {}).get(k), v) for k, v in CKPT_META.items() if ck.get("meta", {}).get(k) != v}
    assert not mismatch, f"best.pt was trained under a different setup: {mismatch}"
    model.load_state_dict(ck["model"]); best_epoch = ck["epoch"]; history = ck.get("history", [])
    LOADED = True
    print(f"loaded {OUT_DIR / 'best.pt'} (epoch {best_epoch}, val {ck['val_loss']:.5f}); the training loop below runs zero epochs")

In [ ]:
# Train the conditional noise predictor, with 10% class-label dropout for CFG.
lr = 2e-4
drop_class_percentage = 0.1
num_epochs = 0 if LOADED else EPOCHS
patience = PATIENCE
min_delta = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
loss_fn = nn.MSELoss()
best_val = significant_best = float("inf")
stale_epochs = 0
best_weights = None


@torch.no_grad()
def Validation_MSE(model):
    model.eval()
    # Reuse the same corruption draws at each epoch for a fair comparison.
    rng = torch.Generator(device=device).manual_seed(seed + 1000)
    total_squared_error = num_pixels = 0
    for x0, labels, captions in val_loader:
        x0, labels = x0.to(device), labels.to(device)
        epsilon = torch.randn(x0.shape, device=device, generator=rng)
        k = torch.randint(K, (len(x0),), device=device, generator=rng)
        alpha = alpha_bars[k][:, None, None, None]
        xk = alpha.sqrt() * x0 + (1 - alpha).sqrt() * epsilon
        predicted = model(xk, k, labels)
        total_squared_error += F.mse_loss(predicted.float(), epsilon, reduction="sum").item()
        num_pixels += epsilon.numel()
    return total_squared_error / num_pixels


_t_start = time.time()
for epoch in range(num_epochs):
    _t_epoch = time.time()
    model.train()
    total_loss = num_images = 0
    for x0, labels, captions in train_loader:
        x0, labels = x0.to(device), labels.to(device)
        k = torch.randint(K, (len(x0),), device=device)  # One timestep 0..999 per image.
        xk, epsilon = Add_Noise(x0, k)
        dropped = torch.rand(len(x0), device=device) < drop_class_percentage
        model_labels = labels.masked_fill(dropped, null_class)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            predicted_noise = model(xk, k, model_labels)
            loss = loss_fn(predicted_noise.float(), epsilon.float())
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(x0)
        num_images += len(x0)

    val_mse = Validation_MSE(model)
    if val_mse < best_val:
        best_val = val_mse
        best_epoch = epoch + 1                                                         # added
        # Keep CPU copies so later updates do not change the saved best model.
        best_weights = {name: tensor.detach().cpu().clone()
                        for name, tensor in model.state_dict().items()}
    if val_mse < significant_best - min_delta:
        significant_best, stale_epochs = val_mse, 0
    else:
        stale_epochs += 1
    history.append(dict(epoch=epoch + 1, train_loss=total_loss / num_images, val_loss=val_mse,
                        seconds=time.time() - _t_epoch))                                # added
    print(f"Epoch {epoch + 1}: train MSE={total_loss / num_images:.5f}, "
          f"validation MSE={val_mse:.5f}, best={best_val:.5f}")
    if patience and stale_epochs >= patience:
        print(f"Early stopping after {epoch + 1} epochs")
        break
TRAIN_SECONDS = time.time() - _t_start if num_epochs else 0.0

if best_weights is not None:
    model.load_state_dict(best_weights)  # Use the validation-selected model below.

## Save the shared checkpoint

Their `fashion_diffusion_best.pt`, plus `best.pt` in the format the three comparison notebooks load, with the setup it was trained under.

In [ ]:
# Optional: save the trained (validation-selected) model.
torch.save(model.state_dict(), "fashion_diffusion_best.pt")

if LOADED:
    best_val = ck["val_loss"]
else:
    best_val = Validation_MSE(model) if best_weights is None else best_val
    checkpoint = dict(model=model.state_dict(), epoch=best_epoch, val_loss=best_val, meta=CKPT_META,
                      code_version=GIT_SHA, history=history,
                      config=dict(lr=lr, batch_size=batch_size, epochs=EPOCHS, patience=patience, drop_class_percentage=drop_class_percentage))
    torch.save(checkpoint, OUT_DIR / "best.pt")
    print(f"saved {OUT_DIR / 'best.pt'} (epoch {best_epoch}, val {best_val:.5f}, {N_PARAMS:,} params)")
model.eval()
print(f"Best validation MSE: {best_val:.5f} at epoch {best_epoch}")

## Sampling with classifier-free guidance

Theirs, with one term added. Their sampler is the deterministic DDIM step over `sample_steps`
steps, the clean estimate clamped to `[-1, 1]`, two network evaluations per step whenever the
scale is not 0 or 1. The added `eta` term is the DDIM paper's eq. 16: the step keeps
`sqrt(1 - abar' - sigma^2) eps` and adds `sigma * noise` with
`sigma = eta * sqrt((1 - abar') / (1 - abar) * (1 - abar / abar'))`. With `eta = 1` this is
ancestral DDPM sampling, the sampler every notebook in this repo uses; with `eta = 0` the added
lines vanish and the function is theirs exactly. The comparison notebooks use the same step,
clamp, 50 steps and `eta`, with two small differences: their step grid is truncated rather than
rounded, and they recompute the noise estimate from the clamped clean estimate before stepping
(the guided sampler needs this). Both only matter where the clamp engages.

In [ ]:
# Reverse diffusion: DDIM-style steps with classifier-free guidance; eta adds the ancestral noise (added).
@torch.no_grad()
def Sample_Images(model, class_labels, guidance_scale=3.0, generator=None, sample_steps=50, eta=None):
    eta = ETA if eta is None else eta
    model.eval()
    class_labels = torch.as_tensor(class_labels, device=device, dtype=torch.long)
    if not 1 <= sample_steps <= K:
        raise ValueError("sample_steps must be between 1 and K")
    if any(label.item() not in active_class_ids for label in class_labels):
        raise ValueError("A requested class has no training images")
    batch = len(class_labels)
    null_labels = torch.full_like(class_labels, null_class)
    xk = torch.randn(batch, 3, image_size, image_size, device=device, generator=generator)
    steps = torch.linspace(K - 1, 0, sample_steps, device=device).round().long().unique_consecutive()

    for index, step in enumerate(steps):
        k = step.expand(batch)
        if guidance_scale == 0:
            epsilon = model(xk, k, null_labels)
        elif guidance_scale == 1:
            epsilon = model(xk, k, class_labels)
        else:
            # Batch conditional and unconditional predictions in one forward pass.
            pair = model(torch.cat([xk, xk]), torch.cat([k, k]),
                         torch.cat([null_labels, class_labels]))
            unconditioned, conditioned = pair.chunk(2)
            epsilon = unconditioned + guidance_scale * (conditioned - unconditioned)

        alpha = alpha_bars[step]
        estimated_x0 = ((xk - (1 - alpha).sqrt() * epsilon) / alpha.sqrt()).clamp(-1, 1)
        if index + 1 == len(steps):
            xk = estimated_x0
        else:
            next_alpha = alpha_bars[steps[index + 1]]
            sigma = eta * ((1 - next_alpha) / (1 - alpha) * (1 - alpha / next_alpha)).sqrt()      # added: DDIM eq. 16
            xk = next_alpha.sqrt() * estimated_x0 + (1 - next_alpha - sigma ** 2).clamp(min=0).sqrt() * epsilon
            if eta > 0:                                                                            # added
                xk = xk + sigma * torch.randn(xk.shape, device=device, generator=generator)
    return xk.clamp(-1, 1)  # [B, 3, 64, 64]

## Experiment 1. Diffusion baseline evaluation

Theirs. FID (torchmetrics, Inception pool features) between `N_EVAL` generated images and the
validation images, first **without** guidance. With 2048 images on each side FID is biased upward;
the counts are recorded with every number.

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance

# Keep real and generated sample counts equal for the FID comparison.
num_eval = min(N_EVAL, len(val_data))  # 2,048 held-out DeepFashion validation images.
eval_batch_size = 16
eval_seed = 123
sample_steps = SAMPLE_STEPS
reference_labels = [label for _, label, _ in val_data.records][:num_eval]
if num_eval < 2048:
    print(f"NOTE: {num_eval} images; FID is biased upward and accuracy is noisy at this count. Not a result.")

In [ ]:
# Baseline FID: generate without class guidance and compare to held-out fashion images.
def To_FID_Format(images):
    '''FID's Inception implementation expects RGB uint8 pixels in [0, 255].'''
    return ((images.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)


def new_fid():
    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)
    fid.set_dtype(torch.float64)
    with torch.no_grad():
        for real_images, _, captions in val_loader:
            fid.update(To_FID_Format(real_images.to(device)), real=True)
    return fid

FID_SCORES, SAMPLE_SECONDS = {}, {}
fid = new_fid()
generator = torch.Generator(device=device).manual_seed(eval_seed)
if device.type == "cuda":
    torch.cuda.synchronize()
start_time = time.perf_counter()

for start in range(0, num_eval, eval_batch_size):
    count = min(eval_batch_size, num_eval - start)
    labels = torch.full((count,), active_class_ids[0], device=device, dtype=torch.long)
    generated = Sample_Images(model, labels, guidance_scale=0.0,
                              generator=generator, sample_steps=sample_steps)
    fid.update(To_FID_Format(generated), real=False)

if device.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start_time
FID_SCORES["0"] = fid.compute().item(); SAMPLE_SECONDS["0"] = elapsed / num_eval
print(f"Validation FID without guidance: {FID_SCORES['0']:.2f}")
print(f"Sampling time: {elapsed:.1f} seconds for {num_eval} images")
print(f"Model evaluations per image: {sample_steps}")

## Experiment 2. Classifier-free guidance evaluation

Theirs. Same validation images and generation seeds, labels drawn with the validation set's class
frequencies, at each guidance strength in `GUIDANCE_SWEEP` other than 0.

In [ ]:
# Use the same class frequencies as the held-out reference images.
labels = torch.tensor(reference_labels, device=device, dtype=torch.long)

for guidance_scale in [w for w in GUIDANCE_SWEEP if w != 0]:
    guided_fid = new_fid()
    generator = torch.Generator(device=device).manual_seed(eval_seed)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start_time = time.perf_counter()
    for start in range(0, num_eval, eval_batch_size):
        batch_labels = labels[start:start + eval_batch_size]
        generated = Sample_Images(model, batch_labels, guidance_scale=guidance_scale,
                                  generator=generator, sample_steps=sample_steps)
        guided_fid.update(To_FID_Format(generated), real=False)

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time
    FID_SCORES[f"{guidance_scale:g}"] = guided_fid.compute().item(); SAMPLE_SECONDS[f"{guidance_scale:g}"] = elapsed / num_eval
    print(f"Guidance {guidance_scale:g}: validation FID={FID_SCORES[f'{guidance_scale:g}']:.2f}")
    print(f"Sampling time: {elapsed:.1f} seconds; model evaluations per image: "
          f"{sample_steps if guidance_scale == 1 else 2 * sample_steps}")

## Samples

Theirs: sixteen categories that have training images, guidance 3.

In [ ]:
# Show samples from sixteen category IDs that have training images.
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

sample_labels = torch.tensor(active_class_ids[:16], device=device)
sample_generator = torch.Generator(device=device).manual_seed(eval_seed)
samples = Sample_Images(model, sample_labels, guidance_scale=3.0,
                        generator=sample_generator, sample_steps=sample_steps)
display_images = ((samples.clamp(-1, 1) + 1) / 2).cpu()
grid = make_grid(display_images, nrow=4)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.show()
print("classes:", [class_names[i] for i in active_class_ids[:16]])

## Experiment 3. Classifier accuracy (top-1 / top-5)

From the flow-matching twin. No pretrained DeepFashion classifier exists, so an ImageNet-pretrained
ResNet-18 is fine-tuned on the **real training images only** (AdamW 1e-3, cosine annealing, 10
epochs, mixed precision) and cached under `checkpoints/`; the code is in
`src/guidance/eval/image_metrics.py`, theirs apart from taking loaders that yield `(image, index)`.
Every DeepFashion notebook scores with this same network. Top-1: the label the image was
conditioned on is the classifier's highest-scoring class; top-5: it is among the five highest.
Guidance 0 ignores the label, so its accuracy is the chance-level floor; real validation images
are the ceiling.

In [ ]:
from guidance.data.fashion import make_loaders
from guidance.eval.image_metrics import fit_classifier, real_accuracy, top_k_hits, precision_recall_features

# Index-yielding loaders over the same records, for the shared classifier code.
clf_train_loader, clf_val_loader, meta = make_loaders(batch_size=batch_size, seed=seed, limit=LIMIT, val_limit=VAL_LIMIT)
assert meta["labels_val"].tolist() == [label for _, label, _ in val_data.records], "validation subsets differ"
# A fine-tuned scorer already on disk is reused: this folder's checkpoints/, the repo's, data/, or
# FASHION_CLF_DIR. The flow-matching base notebook saves its own as fashion_resnet18_classifier.pt.
CLF_SEARCH = [f"{d}/fashion_resnet18_classifier.pt" for d in
              [os.environ.get("FASHION_CLF_DIR"), "../checkpoints", "../../checkpoints", "data", "../../data"] if d]
classifier = fit_classifier("resnet18", num_classes, clf_train_loader, clf_val_loader, meta["labels_train"].to(device),
                            meta["labels_val"].to(device), device, CKPT_DIR / "fashion_resnet18_classifier.pt",
                            epochs=CLF_EPOCHS, seed=seed, search=CLF_SEARCH)
real_top1, real_top5 = real_accuracy(classifier, clf_val_loader, meta["labels_val"].to(device), device)

# Class fidelity: classify generated images and compare with the label each one was conditioned on.
# Same labels (validation class frequencies), seed and step count as Experiment 2, so the numbers are comparable to FID.
eval_labels = torch.tensor(reference_labels, device=device, dtype=torch.long)
accuracy_results = {"real validation images": (real_top1, real_top5)}
per_class_hits = {}
GENERATED = {}

for guidance_scale in GUIDANCE_SWEEP:
    generator = torch.Generator(device=device).manual_seed(eval_seed)
    hit1_all, hit5_all, images_all = [], [], []
    for start in range(0, num_eval, eval_batch_size):
        batch_labels = eval_labels[start:start + eval_batch_size]
        generated = Sample_Images(model, batch_labels, guidance_scale=guidance_scale,
                                  generator=generator, sample_steps=sample_steps)
        with torch.no_grad():
            hit1, hit5 = top_k_hits(classifier(generated), batch_labels)
        hit1_all.append(hit1)
        hit5_all.append(hit5)
        images_all.append(generated)
    hit1_all, hit5_all = torch.cat(hit1_all), torch.cat(hit5_all)
    accuracy_results[f"generated, guidance {guidance_scale:g}"] = (hit1_all.float().mean().item(),
                                                                   hit5_all.float().mean().item())
    per_class_hits[guidance_scale] = hit1_all.cpu()
    GENERATED[f"{guidance_scale:g}"] = torch.cat(images_all)

print(f"{'Images':<30}{'Top-1':>8}{'Top-5':>8}")
for name, (top1, top5) in accuracy_results.items():
    print(f"{name:<30}{top1:>8.3f}{top5:>8.3f}")
print(f"Guidance 0 ignores the label, so its accuracy is the chance-level floor ({1 / len(active_class_ids):.3f}); real images are the ceiling.")

# Per-class top-1 accuracy at the largest guidance (classes with at least 10 validation images), worst first.
w_max = max(GUIDANCE_SWEEP)
ref = torch.tensor(reference_labels)
rows = []
for class_id in sorted(set(reference_labels)):
    mask = ref == class_id
    if mask.sum() >= 10:
        rows.append((per_class_hits[w_max][mask].float().mean().item(), class_names[class_id], int(mask.sum())))
rows.sort()
print(f"\nLowest per-class top-1 accuracy at guidance {w_max:g}:")
for accuracy, name, count in rows[:5]:
    print(f"  {name} ({count} images): {accuracy:.3f}")
print("Highest:")
for accuracy, name, count in rows[-5:][::-1]:
    print(f"  {name} ({count} images): {accuracy:.3f}")

## Precision and recall

From the flow-matching twin. In the classifier's penultimate feature space, with k = 3 nearest
neighbours: **precision** is the share of generated images inside the real images' manifold
(fidelity); **recall** is the share of real validation images inside the generated manifold
(diversity, coverage). Higher guidance usually raises precision and lowers recall.

In [ ]:
pr_k = 3
with torch.no_grad():
    real_features = torch.cat([classifier.features(images.to(device)) for images, _, _ in val_loader])

pr_results = {}
for guidance_scale in GUIDANCE_SWEEP:
    x = GENERATED[f"{guidance_scale:g}"]
    with torch.no_grad():
        generated_features = torch.cat([classifier.features(x[i:i + 256]) for i in range(0, len(x), 256)])
    pr_results[f"{guidance_scale:g}"] = precision_recall_features(real_features, generated_features, k=pr_k)

print(f"Precision / recall with k={pr_k}, {len(val_data)} real vs {num_eval} generated images")
print(f"{'Guidance':<10}{'Precision':>11}{'Recall':>9}")
for guidance_scale, (precision, recall) in pr_results.items():
    print(f"{guidance_scale:<10}{precision:>11.3f}{recall:>9.3f}")
print("Higher guidance usually raises precision and lowers recall (less diverse samples).")

## Manifest

In [ ]:
import json
report = {
    "method": "classifier-free guidance on DeepFashion 64x64; base from CIS6720_Proj1_DiffusionCode_Modality_1.ipynb",
    "modality": "images (DeepFashion 64x64, class label)",
    "code_version": GIT_SHA,
    "quick_showcase_budget": bool(QUICK),
    "config": dict(batch_size=batch_size, seed=seed, image_size=image_size, base_channels=base_channels, K=K,
                   drop_class_percentage=0.1, epochs=EPOCHS, patience=PATIENCE, lr=2e-4, limit=LIMIT, val_limit=VAL_LIMIT,
                   sample_steps=SAMPLE_STEPS, eta=ETA, guidance_sweep=GUIDANCE_SWEEP, n_eval=num_eval, eval_seed=eval_seed, clf_epochs=CLF_EPOCHS),
    "conditioning": "class embedding added to the time embedding; null class for CFG",
    "sampler": f"DDIM step with eta = {ETA:g} ({'ancestral DDPM' if ETA > 0 else 'deterministic DDIM'}), x0 clamped to [-1, 1]",
    "best_val_mse": best_val, "best_epoch": best_epoch, "epochs_run": len(history), "history": history,
    "params": N_PARAMS, "train_seconds": TRAIN_SECONDS, "base_loaded_from_checkpoint": LOADED,
    "n_train": len(train_data), "n_val": len(val_data), "active_classes": len(active_class_ids),
    "checkpoint_meta": CKPT_META,
    "fid_per_guidance": FID_SCORES, "fid_n_real": len(val_data), "fid_n_generated": num_eval,
    "seconds_per_sample_per_guidance": SAMPLE_SECONDS,
    "nfe_per_guidance": {f"{w:g}": (SAMPLE_STEPS if w in (0, 1) else 2 * SAMPLE_STEPS) for w in GUIDANCE_SWEEP},
    "scorer": "fine-tuned ResNet-18 on real training images", "scorer_real_val_top1": real_top1, "scorer_real_val_top5": real_top5,
    "accuracy_per_guidance": {f"{w:g}": {"top1": accuracy_results[f"generated, guidance {w:g}"][0],
                                        "top5": accuracy_results[f"generated, guidance {w:g}"][1]} for w in GUIDANCE_SWEEP},
    "precision_recall_per_guidance": {k: {"precision": v[0], "recall": v[1]} for k, v in pr_results.items()},
    "hardware": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "device": str(device),
}
print(json.dumps({k: v for k, v in report.items() if k != "history"}, indent=2))
with open("manifest_cfg_fashion.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- [ ] `n_eval` at 2048; FID at that count is biased upward, say so, and read it with top-1 and precision / recall
- [ ] NFE stated next to each row: `2 x steps` whenever the scale is not 0 or 1
- [ ] Batch size 64 (not the 16 of the notebook as received) and `eta = 1` (ancestral sampling, not their deterministic sampler) stated with the recipe
- [ ] Three sampling seeds on one checkpoint, mean +/- std, stated as such
- [ ] Training time, epochs actually run (early stopping), parameter count and hardware from the manifest
- [ ] The comparison notebooks loaded this `best.pt`: check `base_loaded_from_checkpoint` there